# TP1 — Pré-traitements avec spaCy

**Master 2 TAL — Notebook étudiant**

Ce notebook constitue le squelette du TP. Les cellules de préparation sont fournies ; les cellules **À COMPLÉTER** correspondent au travail demandé.


## Organisation

- **Question 1** — exploration des données
- **Question 2** — mise en place du pipeline
- **Question 3** — normalisation
- **Question 4** — mots-outils (stopwords)
- **Question 5** — ordre des traitements
- **Question 6** — dépendances
- **Question 7** — composant personnalisé
- **Question 8** — règles vs information linguistique
- **Question 9** — analyse globale des corpus

Les réponses doivent être écrites dans le notebook sous les cellules correspondantes.


## 0. Installation et importations

In [14]:
from logging import disable
!pip -q install spacy nltk pandas
!python -m spacy download fr_core_news_md
!python -m spacy download en_core_web_md


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 21.7 MB/s  0:00:02 eta 0:00:01

[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 5.8 MB/s  0:00:05m0:00:0100:01m

[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [15]:
import spacy
import time
import re
import zipfile
from pathlib import Path
from collections import Counter

import pandas as pd
from spacy import displacy
from spacy.language import Language
from spacy.matcher import Matcher, PhraseMatcher
import xml.etree.ElementTree as ET

print("spaCy :", spacy.__version__)


spaCy : 3.8.16


In [16]:
nlp_fr = spacy.load("fr_core_news_md")
nlp_en = spacy.load("en_core_web_md")

print("Chargement des modèles spaCy ...")
print("Pipeline français :", nlp_fr.pipe_names)
print("Pipeline anglais  :", nlp_en.pipe_names)


Chargement des modèles spaCy ...
Pipeline français : ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Pipeline anglais  : ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


## Chargement des données

Une archive contenant les sous-corpus est fournie avec le TP.

**Attention** : l'analyse est à faire sur les données d'apprentissage.

**À adapter si l’organisation réelle des fichiers est différente.**


In [17]:
# chemin des fichiers fr et en, txt et xml (apprentissage et test)
chemin_txt_fr = "data/deft09_parlement_ref/deft09_parlement_ref_fr.txt"
chemin_txt_en = "data/deft09_parlement_ref/deft09_parlement_ref_en.txt"

chemin_appr_fr = 'data/deft09_parlement_appr.xml/deft09_parlement_appr_fr.xml'
chemin_appr_en = 'data/deft09_parlement_appr.xml/deft09_parlement_appr_en.xml'

chemin_test_fr = 'data/deft09_parlement_test.xml/deft09_parlement_test_fr.xml'
chemin_test_en = 'data/deft09_parlement_test.xml/deft09_parlement_test_en.xml'

print("Chargement de tous les corpus effectué avec succès !")

Chargement de tous les corpus effectué avec succès !


### Chargement des textes

Charger les fichiers `.txt` (attention à séparer le corpus français du corpus italien).


In [18]:
def charger_corpus_txt(chemin_fichier):
    """
    charge un fichier .txt et retourne une liste de textes
    on suppose ici que chaque ligne correspond à un document/paragraphe.
    """
    corpus = []
    with open(chemin_fichier, 'r', encoding='utf-8') as f:
        for ligne in f:
            ligne_propre = ligne.strip()  #
            # on ajoute à la liste corpus les textes si la ligne n'est pas vide
            if ligne_propre:
                corpus.append(ligne_propre)
    return corpus


print("Chargement des textes en cours...")

corpus_fr = charger_corpus_txt(chemin_txt_fr)
corpus_en = charger_corpus_txt(chemin_txt_en)

print(f"Chargement terminé ! \n- {len(corpus_fr)} documents français.\n- {len(corpus_en)} documents anglais.")

# affichage des premiers textes de chaque corpus
if corpus_fr:
    print("\n--- Extrait français ---")
    print(corpus_fr[0:5])

if corpus_en:
    print("\n--- Extrait anglais ---")
    print(corpus_en[0:5])

Chargement des textes en cours...
Chargement terminé ! 
- 12915 documents français.
- 12917 documents anglais.

--- Extrait français ---
['1\tELDR', '2\tGUE-NGL', '3\tPPE-DE', '4\tGUE-NGL', '5\tPPE-DE']

--- Extrait anglais ---
['1\tPSE', '2\tPPE-DE', '3\tVerts-ALE', '4\tPPE-DE', '5\tELDR']


# 1. Question 1 — Première exploration des données

Pour chacun des corpus, fournissez le nombre de documents, de phrases, de mots/tokens, la longueur moyenne des documents, la longueur moyenne des phrases et le nombre de formes distinctes.

Observez également plusieurs exemples de phrases et identifiez au moins deux phénomènes susceptibles de poser problème à un système de TAL.

In [19]:
# ==============================================================
# À COMPLÉTER — 1. Question 1 — Première exploration des données
# ==============================================================

def load_xml_corpus(chemin_fichier):
    """
    extraction de l'intégralité du texte de la balise <texte>
    pour chaque document <doc> du fichier XML
    """
    corpus = []
    arbre = ET.parse(chemin_fichier)
    racine = arbre.getroot()

    # parcours de chaque document dans le corpus
    for doc in racine.findall('.//doc'):
        # on cible directement la balise <texte>
        balise_texte = doc.find('.//texte')

        if balise_texte is not None:
            # récupère tout le texte
            texte_complet = "".join(balise_texte.itertext())
            corpus.append(texte_complet.strip())

    return corpus


corpus_fr = load_xml_corpus(chemin_appr_fr)
corpus_en = load_xml_corpus(chemin_appr_en)

print(f"Chargement terminé : {len(corpus_en)} documents anglais, {len(corpus_fr)} documents français.")


def data_exploration(corpus, nlp):
    nbre_docs = len(corpus)
    nbre_phrases = 0
    nbre_tokens = 0
    formes_distinctes = set()

    for doc in nlp.pipe(corpus, disable=['ner']):
        nbre_phrases += len(list(doc.sents))

        for token in doc:
            nbre_tokens += 1
            formes_distinctes.add(token.text)

    long_moy_doc = nbre_tokens / nbre_docs if nbre_docs > 0 else 0
    long_moy_phrase = nbre_tokens / nbre_phrases if nbre_phrases > 0 else 0

    print(f"  - Nombre de documents : {nbre_docs}")
    print(f"  - Nombre de phrases : {nbre_phrases}")
    print(f"  - Nombre de mots/tokens : {nbre_tokens}")
    print(f"  - Longueur moyenne des documents : {long_moy_doc:.2f} tokens")
    print(f"  - Longueur moyenne des phrases : {long_moy_phrase:.2f} tokens")
    print(f"  - Nombre de formes distinctes : {len(formes_distinctes)}\n")


print("\n--- Statistiques du corpus français --- \n")
data_exploration(corpus_fr, nlp_fr)
print("--- Statistiques du corpus anglais ---\n")
data_exploration(corpus_en, nlp_en)


Chargement terminé : 12917 documents anglais, 12915 documents français.

--- Statistiques du corpus français --- 

  - Nombre de documents : 12915
  - Nombre de phrases : 189490
  - Nombre de mots/tokens : 5149960
  - Longueur moyenne des documents : 398.76 tokens
  - Longueur moyenne des phrases : 27.18 tokens
  - Nombre de formes distinctes : 53206

--- Statistiques du corpus anglais ---

  - Nombre de documents : 12917
  - Nombre de phrases : 172137
  - Nombre de mots/tokens : 4764453
  - Longueur moyenne des documents : 368.85 tokens
  - Longueur moyenne des phrases : 27.68 tokens
  - Nombre de formes distinctes : 39908



A l'issu de ces résultats, on peut remarquer que le corpus en français comporte une très grande variété de formes distinctes (vocabulaire) que l'anglais. Ceci explique logiquement la complexité de la morphologie (accords genre/nombre, conjugaison).

In [20]:
print(corpus_fr[:50])

["Monsieur le Président, l'accès à des médias libres dans un débat ouvert et libre est l'une des pierres angulaires d'une société libre et démocratique. Les dictateurs de par le monde l'ont toujours su. Toutes les sociétés, tous les États qui répriment leur propre presse ne recherchent qu'une seule chose : faire taire le débat et asphyxier la démocratie, la communication ouverte et libre.\n   Cela se manifeste tous les jours en Serbie. Les quelques stations de télévision et de radios indépendantes serbes subissent des tracasseries et sont pourchassées, pour finir par être totalement fermées. M. Milosevic menace le Monténégro, il s'en prend à la faible opposition serbe, il s'infiltre au Kosovo. M. Milosevic et ses complices doivent s'en aller, la démocratie doit être sauvegardée.\n   Je voudrais dès lors poser une question au commissaire Nielson. Que fait-il et que fait l'UE pour faire en sorte que M. Milosevic soit traduit en justice devant la Cour internationale de justice de La Haye.

In [21]:
print(corpus_en[:20])

['Mr President, I would like to propose an oral amendment concerning paragraph 34. In fact, I shall not propose any fundamental changes, as it is just a matter of linking several amendments which take account of new considerations, which apply following the tabling of the draft report, namely the peace proposals by the Secretary-General of the United Nations.\n   I would propose keeping paragraph 34 as it appears in the draft text, and adding Amendment No 42 to this text of paragraph 34, by simply inserting the words ‘and welcomes it in this context’. As for the rest of Amendment No 42, it would remain as is; the words ‘in this context’ would be added, in other words, a simple link. The second part of Amendment No 9 by the  could then be considered as an addition, as well as the second part of Amendment No 12 by the , which would also be an addition. These amendments are compatible. \xa0\n   The President established that at least 32 Members objected to the oral amendment.', 'Madam Pre

-   Les très nombreuses élisions : On trouve "l’oratrice", "L’État", "qu’à", "s’ouvrir". C'est un problème très spécifique au français en TAL. Le système doit être configuré pour comprendre que "l'" est un token (le déterminant) et "oratrice" en est un autre (le nom), et ne pas les garder attachés.

# 2. Question 2 — Mise en place du pipeline

Construisez un pipeline spaCy permettant d’obtenir segmentation en phrases, tokenisation, lemmatisation, POS tagging et analyse en dépendances.

Appliquez-le aux sous-corpus et mesurez le temps d’exécution. Utilisez `nlp.pipe`.

In [12]:
# ============================================================
# À COMPLÉTER — 2. Question 2 — Mise en place du pipeline
# ============================================================

def pipeline_execution_time(corpus, nlp, langue):
    print(f"Analyse du corpus {langue} en cours ...")
    debut = time.time()
    docs_traites = list(nlp.pipe(corpus))

    fin = time.time()

    temps_execution = fin - debut

    print(f"Temps execution : {temps_execution:.2f} secondes pour {len(docs_traites)} documents {langue} \n")


pipeline_execution_time(corpus_fr, nlp_fr, 'français')
pipeline_execution_time(corpus_en, nlp_en, 'anglais')

Analyse du corpus français en cours ...
Temps execution : 264.43 secondes pour 12915 documents français 

Analyse du corpus anglais en cours ...
Temps execution : 276.14 secondes pour 12917 documents anglais 



In [13]:
len(corpus_fr)

12915

In [22]:
def pipeline_chronometer(corpus, nlp, nom_langue):
    print(f"Analyse du corpus {nom_langue} en cours...")
    debut = time.time()

    nb_docs = 0
    # on utilise un batch_size pour optimiser le traitement par lots
    for doc in nlp.pipe(corpus, batch_size=100):
        nb_docs += 1

    fin = time.time()
    temps_execution = fin - debut

    print(f"Temps d'exécution : {temps_execution:.2f} secondes pour {nb_docs} documents.\n")


pipeline_chronometer(corpus_fr, nlp_fr, "Français")
pipeline_chronometer(corpus_en, nlp_en, "Anglais")

Analyse du corpus Français en cours...
Temps d'exécution : 252.23 secondes pour 12915 documents.

Analyse du corpus Anglais en cours...
Temps d'exécution : 277.15 secondes pour 12917 documents.



Deux fonctions ont été implémentée pour faire un comparatif de celle qui serait la plus rapide. A l'issue, on peut remarquer que le fait de prendre les

# 3. Question 3 — Normalisation

Comparez les représentations suivantes : texte original, texte en minuscules, texte lemmatisé et texte soumis à une racinisation (*stemming*).

Pour chaque représentation, calculez le nombre de formes distinctes. Analysez les différences et donnez quelques exemples de regroupements corrects ou incorrects.

In [23]:
# ============================================================
# À COMPLÉTER — 3. Question 3 — Normalisation
# ============================================================

from nltk.stem.snowball import SnowballStemmer

stemmer_fr = SnowballStemmer(language='french')
stemmer_en = SnowballStemmer(language='english')


def corpus_normalization(corpus, nlp, stemmer, langue):
    print(f"--- Analyse de la normalisation : Corpus {langue} --- \n")

    original_form = set()
    minuscule_form = set()
    lemmes_form = set()
    stems_form = set()

    for doc in nlp.pipe(corpus[:1000], disable=['ner', 'parser']):
        for token in doc:
            if not token.is_punct and not token.is_space:
                mot = token.text

                original_form.add(mot)
                minuscule_form.add(mot.lower())
                lemmes_form.add(token.lemma_)
                stems_form.add(stemmer.stem(mot))

    print(f"Formes distinctes, texte original => {len(original_form)}")
    print(f"Formes distinctes, texte en minuscules => {len(minuscule_form)}")
    print(f"Formes distinctes, texte lemmatisé => {len(lemmes_form)}")
    print(f"Formes distinctes, texte racinisé (stemming)) => {len(stems_form)}\n")


corpus_normalization(corpus_fr, nlp_fr, stemmer_fr, "Français")
corpus_normalization(corpus_en, nlp_en, stemmer_en, "Anglais")


--- Analyse de la normalisation : Corpus Français --- 

Formes distinctes, texte original => 18696
Formes distinctes, texte en minuscules => 17975
Formes distinctes, texte lemmatisé => 10974
Formes distinctes, texte racinisé (stemming)) => 9745

--- Analyse de la normalisation : Corpus Anglais --- 

Formes distinctes, texte original => 14366
Formes distinctes, texte en minuscules => 13184
Formes distinctes, texte lemmatisé => 10768
Formes distinctes, texte racinisé (stemming)) => 8364



In [24]:
# initialisation du dictionnaire global
conflits_stemming_fr = {}
conflits_stemming_en = {}

# définition de la fonction de remplissage
def remplir_dictionnaire(corpus, nlp, stemmer, conflits_stemming):
    for doc in nlp.pipe(corpus[:10], disable=['ner', 'parser']):
        for token in doc:
            if token.is_alpha and len(token.text) > 3:
                mot = token.text.lower()  # on sécurise en minuscules
                lemme = token.lemma_.lower()
                racine = stemmer.stem(mot)

                if racine not in conflits_stemming:
                    conflits_stemming[racine] = set()
                conflits_stemming[racine].add((mot, lemme))

remplir_dictionnaire(corpus_fr, nlp_fr, stemmer_fr, conflits_stemming_fr)
remplir_dictionnaire(corpus_en, nlp_en, stemmer_en, conflits_stemming_en)


#print(f"{'Mot original':<15} | {'Lemme (spaCy)':<15} | {'Racine (NLTK)':<15} | Statut")
#print("-" * 65)

def correct_incorrect_grouping(conflits_stemming):
    compteur_erreurs = 0
    compteur_succes = 0

    for racine, couples in conflits_stemming.items():

        lemmes_uniques = {lemme for mot, lemme in couples}

        # regroupement INCORRECT
        if len(lemmes_uniques) > 1 and compteur_erreurs < 5:
            print(f"\n*** Racine tronquée {racine.upper()} fusionne à tort ***")
            print("-" * 65)
            print(f"{'Mot original':<15} | {'Lemme':<15} | {'Racine':<15} | Statut")
            print("-" * 65)
            for mot, lemme in couples:
                print(f"{mot:<15} | {lemme:<15} | {racine:<15} | Incorrect")
            print("-" * 65)
            compteur_erreurs += 1

        # regroupement CORRECT (un seul lemme, mais plusieurs mots différents)
        elif len(lemmes_uniques) == 1 and len(couples) > 1 and compteur_succes < 5:
            print(f"\n*** Regroupement correct sous la racine {racine.upper()} ***")
            print("-" * 65)
            print(f"{'Mot original':<15} | {'Lemme':<15} | {'Racine':<15} | Statut")
            print("-" * 65)
            for mot, lemme in couples:
                print(f"{mot:<15} | {lemme:<15} | {racine:<15} | Correct")
            print("-" * 65)
            compteur_succes += 1

print("--- Analyse des regroupements du stemming sur le corpus FRANÇAIS ---")
correct_incorrect_grouping(conflits_stemming_fr)

print("\n--- Analyse des regroupements du stemming sur le corpus ANGLAIS ---")
correct_incorrect_grouping(conflits_stemming_en)

--- Analyse des regroupements du stemming sur le corpus FRANÇAIS ---

*** Racine tronquée LIBR fusionne à tort ***
-----------------------------------------------------------------
Mot original    | Lemme           | Racine          | Statut
-----------------------------------------------------------------
libres          | libre           | libr            | Incorrect
libre           | libre           | libr            | Incorrect
librement       | librement       | libr            | Incorrect
-----------------------------------------------------------------

*** Regroupement correct sous la racine OUVERT ***
-----------------------------------------------------------------
Mot original    | Lemme           | Racine          | Statut
-----------------------------------------------------------------
ouvertes        | ouvert          | ouvert          | Correct
ouvert          | ouvert          | ouvert          | Correct
ouverte         | ouvert          | ouvert          | Correct
---

# 4. Question 4 — Mots-outils (stopwords)

Supprimez les mots vides et comparez les statistiques avant/après suppression. Donnez un exemple de mot dont la suppression vous paraît pertinente et un exemple pour lequel elle pourrait être discutable.

In [25]:
# ============================================================
# À COMPLÉTER — 4. Question 4 — Mots-outils (stopwords)
# ============================================================

def destroy_stop_words(corpus, nlp, langue):

    total_token = 0
    useful_word = 0
    stop_words = []

    for doc in nlp.pipe(corpus[:10], disable=['ner', 'parser']):
        for token in doc:
            if not token.is_punct and not token.is_space:
                total_token += 1

                if not token.is_stop:
                    useful_word += 1
                else:
                    stop_words.append(token.text)

    remaining_percent = (useful_word / total_token) * 100
    print(stop_words)
    print("-" * 19)
    print(f"* CORPUS {langue.upper()} *")
    print("-" * 19, "\n")
    print(f"Nombre total de mots (hors ponctuation) avant suppression : {total_token}")
    print(f"Nombre total de mots après suppression : {useful_word}")
    print(f"Pourcentage de mots conservé : {remaining_percent:.2f}%\n")


destroy_stop_words(corpus_fr, nlp_fr, "Français")
destroy_stop_words(corpus_en, nlp_en, "Anglais")


['le', "l'", 'à', 'des', 'dans', 'un', 'ouvert', 'et', 'est', "l'", 'une', 'des', "d'", 'une', 'et', 'Les', 'de', 'par', 'le', "l'", 'ont', 'toujours', 'Toutes', 'les', 'tous', 'les', 'qui', 'leur', 'ne', "qu'", 'une', 'seule', 'le', 'et', 'la', 'la', 'ouverte', 'et', 'Cela', 'se', 'tous', 'les', 'en', 'Les', 'quelques', 'de', 'et', 'de', 'des', 'et', 'sont', 'pour', 'par', 'être', 'le', 'il', "s'", 'en', 'à', 'la', 'il', "s'", 'au', 'et', 'ses', 'doivent', "s'", 'en', 'la', 'doit', 'être', 'Je', 'dès', 'lors', 'une', 'au', 'Que', 'fait', 'et', 'que', 'fait', "l'", 'pour', 'en', 'que', 'soit', 'en', 'devant', 'la', 'de', 'de', 'La', "C'", 'est', 'une', 'pour', "l'", 'que', 'les', 'de', 'de', "n'", 'pas', 'devant', 'un', 'et', "qu'", 'il', 'à', "s'", 'en', 'à', 'la', 'à', 'la', 'à', 'la', 'et', 'à', 'la', 'de', 'la', 'le', "j'", 'ai', 'un', 'de', 'différent', 'Je', 'par', 'comme', 'les', 'dans', 'le', 'd’', 'un', 'des', 'et', 'pour', 'l’', 'et', 'l’', 'sur', 'l’', 'des', 'dans', 'la', '

Le fait que les deux corpus aboutissent à une réduction quasi identique prouve que la structure syntaxique du discours formel est très similaire dans ces deux langues. Le français utilise légèrement plus de petits mots articulatoires (comme "de", "la", "au"), ce qui explique une chute très légèrement plus prononcée de son volume (43,39 % conservés contre 44,21 % en anglais).

L'avantage immédiat de ce filtrage pour la suite de ton analyse est double :

* Gain de performances : Tu viens d'éliminer plus de 200 000 jetons côté français. Les prochains calculs seront beaucoup plus rapides.
* Gain de pertinence sémantique : En supprimant le bruit grammatical, les algorithmes pourront se concentrer exclusivement sur les mots porteurs de sens (les sujets des débats). Cela est indispensable avant de passer à l'étape suivante, comme la création d'une représentation vectorielle du texte (Bag-of-Words ou TF-IDF).

Un exemple de suppression pertinente : "et", "le" ou "de"
Pourquoi c'est pertinent : Ces mots (conjonctions, articles, prépositions de base) sont ce qu'on appelle du "bruit" thématique. Qu'un discours parlementaire contienne 500 fois le mot "le" ou "de" ne donne strictement aucune information à la machine sur le sujet du débat (écologie, économie, santé...).

L'argument pour ton rapport : Leur suppression allège considérablement la matrice de données sans aucune perte de sens pour un algorithme qui chercherait à classer les textes par thèmes.

Un exemple de suppression discutable : "pas", "jamais" ou "contre"
Pourquoi c'est discutable : Les listes de stop-words par défaut suppriment souvent les adverbes de négation ("pas", "ne", "aucun") ou certaines prépositions ("pour", "contre"). Dans un contexte de débat parlementaire, c'est extrêmement dangereux !

L'argument pour ton rapport : Si tu supprimes la négation "pas", la phrase "Nous ne voterons pas cette loi" devient pour l'algorithme "nous voterons loi". Le sens de la phrase est diamétralement inversé ! De la même manière, effacer les pronoms ("nous" vs "vous") détruit l'information sur l'opposition rhétorique entre les partis politiques.

L'utilisation d'une liste de stop-words générique (comme celle par défaut de spaCy) n'est jamais neutre. L'expert en TAL doit toujours adapter et personnaliser cette liste en fonction du domaine de son corpus (ici, la politique) pour ne pas détruire d'informations vitales.

In [26]:
def prouver_impact_suppression(corpus, nlp, mot_cible, fenetre=6):
    print(f"--- Impact de la suppression du mot '{mot_cible}' ---")
    compteur = 0

    for doc in nlp.pipe(corpus[:500], disable=["ner", "parser"]):
        for i, token in enumerate(doc):
            if token.text.lower() == mot_cible:
                # On capture les mots juste avant et juste après pour le contexte
                debut = max(0, i - fenetre)
                fin = min(len(doc), i + fenetre + 1)

                # On reconstruit le bout de phrase
                contexte = " ".join([t.text for t in doc[debut:fin]])
                print(f"Extrait {compteur + 1} : ... {contexte} ...")

                compteur += 1
                if compteur >= 3: # On s'arrête après 3 exemples probants
                    return

# On teste avec un mot dont la suppression est discutable
prouver_impact_suppression(corpus_fr, nlp_fr, "pas")
prouver_impact_suppression(corpus_fr, nlp_fr, "contre")

--- Impact de la suppression du mot 'pas' ---
Extrait 1 : ... de guerre de Milosevic n' aient pas été jugés devant un tribunal et ...
Extrait 2 : ... valable . Il n’ en reste pas moins que je considère les positions ...
Extrait 3 : ... a révélé que nous n' avions pas transféré à la production agricole primaire ...
--- Impact de la suppression du mot 'contre' ---
Extrait 1 : ... droit soit écarté dans la lutte contre le terrorisme . Si vous écartez ...
Extrait 2 : ... existe quant la question du christianisme contre l’ islam . C’ est inacceptable ...
Extrait 3 : ... à l' agression barbare et criminelle contre le peuple yougoslave . Elle le ...


# 5. Question 5 — Ordre des traitements

Construisez et comparez :

- Pipeline A : texte → tokenisation → POS tagging
- Pipeline B : texte → tokenisation → normalisation → POS tagging

Les annotations sont-elles identiques ? Expliquez les différences éventuelles et indiquez où vous placeriez la normalisation dans un pipeline destiné à produire des annotations linguistiques.

> Le tokenizer spaCy intervient avant les autres composants. Pour cette expérience, vous pouvez créer une représentation normalisée du texte puis la transmettre à un second pipeline, en explicitant cette limite.

In [75]:
from spacy.tokens import Doc

def pipeline_a(corpus, nlp):
    return list(nlp.pipe(corpus, disable=["lemmatizer", "parser", "ner"]))

def pipeline_b(corpus, nlp):
    normalized_corpus = []
    for texte in corpus:
        # tokenisation initiale
        doc_initial = nlp.make_doc(texte)
        # normalisation des tokens
        tokens_normalises = [token.text.lower() for token in doc_initial]
        # création de la représentation normalisée à transmettre au second pipeline
        #texte_normalise = " ".join(tokens_normalises)
        les_espaces = [token.whitespace_ for token in doc_initial]
        # reconstruction parfaite avec l'objet Doc de spacy
        texte_normalise = Doc(nlp.vocab, words=tokens_normalises, spaces=[bool(e) for e in les_espaces])
        normalized_corpus.append(texte_normalise)

    return list(nlp.pipe(normalized_corpus))

# on stocke les résultats dans des variables
pipeline_a_fr = pipeline_a(corpus_fr, nlp_fr)
pipeline_b_fr = pipeline_b(corpus_fr, nlp_fr)

pipeline_a_en = pipeline_a(corpus_en, nlp_en)
pipeline_b_en = pipeline_b(corpus_en, nlp_en)

In [76]:
def compare_annotations(pipeline_a, pipeline_b, langue):
    total_mots = 0
    differences_pos = 0
    example_pos = []

    print(f"\n--- Comparaison Pipeline A vs B ({langue}) ---")

    for doc_a, doc_b in zip(pipeline_a, pipeline_b):
        # Vérification de l'alignement des tokens
        if len(doc_a) == len(doc_b):
            for token_a, token_b in zip(doc_a, doc_b):
                total_mots += 1

                # Comptage des différences d'annotations
                if token_a.pos_ != token_b.pos_:
                    differences_pos += 1

                    # Affichage des 3 premiers cas où un nom propre (PROPN) est altéré
                    if token_a.pos_ not in ["PUNCT", "SPACE"] and differences_pos > 5:
                        example_pos.append((token_a.text, token_a.pos_, token_b.pos_))

    print("-" * 50)
    print(f"{"Mot original":<15} | {"Pipeline A":<15} | {"Pipeline B"}")
    print("-" * 50)
    for diff in example_pos[:10]:
        print(f"{diff[0]:<15} | {diff[1]:<15} | {diff[2]}")

    pourcentage_diff = (differences_pos / total_mots) * 100 if total_mots > 0 else 0
    print(f"\nTotal des étiquettes POS modifiées : {differences_pos} sur {total_mots} tokens ({pourcentage_diff:.2f}%)")

compare_annotations(pipeline_a_fr, pipeline_b_fr, "Français")
compare_annotations(pipeline_a_en, pipeline_b_en, "Anglais")


--- Comparaison Pipeline A vs B (Français) ---
--------------------------------------------------
Mot original    | Pipeline A      | Pipeline B
--------------------------------------------------
-               | NOUN            | PUNCT
membres         | NOUN            | ADJ
HACCP           | NOUN            | ADJ
fait            | VERB            | NOUN
-               | NOUN            | PUNCT
-               | PROPN           | PUNCT
Kindermann      | PROPN           | NOUN
membres         | NOUN            | ADJ
Caire           | PROPN           | NOUN
Aujourd’hui     | ADV             | NOUN

Total des étiquettes POS modifiées : 59075 sur 5149960 tokens (1.15%)

--- Comparaison Pipeline A vs B (Anglais) ---
--------------------------------------------------
Mot original    | Pipeline A      | Pipeline B
--------------------------------------------------
as              | ADP             | SCONJ
Amendment       | PROPN           | NOUN
Amendment       | PROPN           | NOUN
Am

**1- Les annotations sont-elles identiques ?**

Non, les annotations ne sont pas identiques. Le test sur l'intégralité des données montre une altération de `1,15 %` des étiquettes en français (près de 60 000 tokens) et de `5,70 %` en anglais (plus de 270 000 tokens).

**2- Explication des différences éventuelles :**

Le modèle d'étiquetage morpho-syntaxique (POS tagging) utilise la casse (les majuscules) comme indice principal pour détecter les entités nommées. En appliquant une normalisation en minuscules avant l'étiquetage (Pipeline B), on détruit cette information. Le modèle est alors aveuglé et rétrograde à tort les noms propres (`PROPN`) en noms communs (`NOUN`), comme le prouvent les déclassements de "Kindermann" et "Caire" en français, ou de "Amendment" et "Vice" en anglais.

**3- Où placer la normalisation ?**

Pour un pipeline destiné à produire des annotations linguistiques, la normalisation doit impérativement être placée à la fin, c'est-à-dire après la tokenisation, le POS tagging et l'analyse en dépendances. Il faut laisser les modèles d'annotation travailler sur le texte brut d'origine.

In [ ]:
def comparer_pipelines_sur_corpus(corpus, nlp, langue, nb_docs=10):
    print(f"\n{'='*60}")
    print(f"COMPARAISON PIPELINE A vs B - CORPUS {langue.upper()}")
    print(f"{'='*60}")

    total_tokens = 0
    erreurs_pos = 0
    no_match = 0
    exemples_propn = []

    # ==========================================
    # PRÉPARATION DU PIPELINE B (Limite explicitée)
    # texte → tokenisation → normalisation → [texte normalisé] → POS tagging
    # ==========================================


    print("\n--- VISUALISATION DES DÉCALAGES DE TOKENISATION ---")
    nb_exemples_a_afficher = 0
    exemples_affiches = 0

    for i, (doc_a, doc_b) in enumerate(zip(pipeline_A, pipeline_B)):
        # Si les deux pipelines n'ont pas découpé le texte de la même façon
        tokens_a = [t.text for t in doc_a]
        # Liste des mots découpés par le Pipeline B (Après le .join())
        tokens_b = [t.text for t in doc_b]

        # print(f"Pipeline A : {tokens_a}")
        # print(f"Pipeline B : {tokens_b}")

    # ==========================================
    # COMPARAISON DES ANNOTATIONS
    # ==========================================
    for doc_a, doc_b in zip(pipeline_A, pipeline_B):
        # On compare uniquement si la re-tokenisation n'a pas décalé les mots
        if len(doc_a) == len(doc_b):
            for token_a, token_b in zip(doc_a, doc_b):
                total_tokens += 1

                # Si l'annotation POS est différente entre A et B
                if token_a.pos_ != token_b.pos_:
                    erreurs_pos += 1

                    # On capture quelques exemples où un nom propre a été détruit par le Pipeline B
                    if token_a.pos_ not in ["PUNCT", "SPACE"] and len(exemples_propn) < 5:
                        exemples_propn.append((token_a.text, token_a.pos_, token_b.pos_))
        else:
            no_match =+ 1
    pourcentage = (erreurs_pos / total_tokens) * 100 if total_tokens > 0 else 0


    print(f"Tokens analysés : {no_match}")
    print(f"Tokens analysés : {total_tokens}")
    print(f"Étiquettes POS modifiées : {erreurs_pos} (soit {pourcentage:.2f}% de différence)\n")

    print("--- Exemples concrets extraits de vos données ---")
    #print("Des noms propres (PROPN) du Pipeline A sont devenus :")
    print("-" * 65)
    print(f"{"Mot original":<15} | {"Pipeline A":<15} | {"Pipeline B"}")
    print("-" * 65)
    for diff in exemples_propn:
        print(f"{diff[0]:<15} | {diff[1]:<15} | {diff[2]}")

# Lancement de l'expérience sur les deux corpus (limité à 100 documents pour la vitesse)
comparer_pipelines_sur_corpus(corpus_fr, nlp_fr, "Français")
#comparer_pipelines_sur_corpus(corpus_en, nlp_en, "Anglais")

# 6. Question 6 — Dépendances

À partir d’une phrase en français, visualisez l’analyse syntaxique avec `displacy`. Sélectionnez une phrase représentative et discutez une analyse qui vous semble surprenante ou potentiellement erronée.

In [88]:
from spacy import displacy

# On analyse spécifiquement la phrase choisie
texte_choisi = "Le ministre propose un amendement pour soutenir les entreprises de la région."
doc_phrase = nlp_fr(texte_choisi)
# Génération du graphe de dépendances dans le notebook
displacy.render(doc_phrase, style="dep", jupyter=True, options={"distance": 130})

Le ministre propose un amendement pour soutenir les entreprises de la région.


# 7. Question 7 — Composant personnalisé

Implémentez un composant de pipeline permettant de réaliser une normalisation à base de règles sur un phénomène présent dans les données (nombres, dates, abréviations, etc.). Le composant devra être intégré avec `nlp.add_pipe`.

Mesurez le temps du pipeline :
- sans votre composant ;
- avec votre composant.

Indiquez son surcoût et expliquez où vous placez votre composant dans le pipeline et pourquoi.

Analysez également son éventuel impact sur les tokens, les lemmes et les POS.


In [92]:
# ============================================================
# À COMPLÉTER — 7. Question 7 — Composant personnalisé
# ============================================================

import time
from spacy.language import Language

# 1. Définition du composant personnalisé
@Language.component("normalisation_parlementaire")
def normalisation_parlementaire(doc):
    # Dictionnaire de règles pour le corpus parlementaire
    regles = {"M.": "monsieur", "Mme": "madame", "art.": "article"}
    for token in doc:
        if token.text in regles:
            # On modifie la représentation normalisée du token
            token.norm_ = regles[token.text]
    return doc

# On prend un échantillon représentatif pour avoir une mesure de temps fiable
echantillon = corpus_fr[:500]

# 2. Mesure du temps SANS le composant
debut_sans = time.time()
docs_sans = list(nlp_fr.pipe(echantillon))
temps_sans = time.time() - debut_sans

print(nlp_fr.pipe_names)

# 3. Intégration du composant (on s'assure de ne pas l'ajouter deux fois)
if "normalisation_parlementaire" not in nlp_fr.pipe_names:
    # On le place au tout début du pipeline (après la tokenisation)
    nlp_fr.add_pipe("normalisation_parlementaire", first=True)

# 4. Mesure du temps AVEC le composant
debut_avec = time.time()
docs_avec = list(nlp_fr.pipe(echantillon))
temps_avec = time.time() - debut_avec

surcout = temps_avec - temps_sans

print(f"Temps sans le composant : {temps_sans:.2f} secondes")
print(f"Temps avec le composant : {temps_avec:.2f} secondes")
print(f"Surcoût du composant    : {surcout:.2f} secondes")

['normalisation_parlementaire', 'tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Temps sans le composant : 11.11 secondes
Temps avec le composant : 11.03 secondes
Surcoût du composant    : -0.08 secondes


# 8. Question 8 — Règles vs information linguistique

Choisissez un phénomène pouvant être détecté par un `Matcher` ou `PhraseMatcher` et à partir d’une information linguistique produite par spaCy, par exemple les POS.

Construisez un petit jeu de test d’au moins 20 phrases, annotées manuellement, en français.

Calculez pour les deux méthodes :
- précision, rappel, F1-mesure ;
- temps d’exécution.

Comparez les deux approches.

Analysez quelques faux positifs et faux négatifs et expliquez les différences de performance.

Dans quelles situations privilégieriez-vous une approche à base de règles ?

In [97]:
# ============================================================
# À COMPLÉTER — 8. Question 8 — Règles vs information linguistique
# ============================================================

import spacy
import time
from spacy.matcher import Matcher

# 1. JEU DE TEST (20 phrases)
# Format : (Texte, Liste des VRAIS noms propres attendus - vérité terrain)
jeu_de_test = [
    ("Emmanuel Macron a visité Paris hier.", ["Emmanuel", "Macron", "Paris"]),
    ("Le président de la République parle ce soir.", ["République"]),
    ("Apple lance un nouveau téléphone.", ["Apple"]),
    ("Les députés de l'Assemblée nationale ont voté.", ["Assemblée"]),
    ("Je vais à Lyon ce weekend.", ["Lyon"]),
    ("Il aime beaucoup la France.", ["France"]),
    ("Demain, Jean et Marie partent en Italie.", ["Jean", "Marie", "Italie"]),
    ("Le journal Le Monde a publié un article fascinant.", ["Monde"]),
    ("L'entreprise Renault recrute.", ["Renault"]),
    ("Victor Hugo est un grand écrivain.", ["Victor", "Hugo"]),
    ("La Seine traverse la capitale.", ["Seine"]),
    ("Berlin est une ville magnifique.", ["Berlin"]),
    ("Le traité de Rome a été signé il y a longtemps.", ["Rome"]),
    ("Sophie regarde un film.", ["Sophie"]),
    ("Les Nations Unies se réunissent à New York.", ["Nations", "Unies", "New", "York"]),
    ("Mon ami Pierre travaille chez Google.", ["Pierre", "Google"]),
    ("La Banque Centrale Européenne ajuste ses taux.", ["Banque", "Centrale", "Européenne"]),
    ("J'ai lu un livre sur Napoléon.", ["Napoléon"]),
    ("Le sommet du G7 aura lieu au Japon.", ["G7", "Japon"]),
    ("Amazon livre des millions de colis.", ["Amazon"])
]

# 2. PRÉPARATION DU MATCHER
matcher = Matcher(nlp_fr.vocab)
# Règle : le mot commence par une majuscule et contient des lettres
pattern = [{"IS_TITLE": True, "IS_ALPHA": True}]
matcher.add("NOMS_PROPRES", [pattern])

# Variables pour stocker les résultats
vrais_positifs_matcher, faux_positifs_matcher, faux_negatifs_matcher = 0, 0, 0
vrais_positifs_pos, faux_positifs_pos, faux_negatifs_pos = 0, 0, 0

# 3. ÉVALUATION ET MESURE DU TEMPS
# --- MÉTHODE 1 : MATCHER ---
debut_matcher = time.perf_counter()
for texte, verite in jeu_de_test:
    doc = nlp_fr.make_doc(texte) # On utilise make_doc pour ne pas tricher (pas de POS tagger caché)
    matches = matcher(doc)
    predictions_matcher = [doc[start:end].text for match_id, start, end in matches]

    # Calcul TP, FP, FN
    for pred in predictions_matcher:
        if pred in verite: vrais_positifs_matcher += 1
        else: faux_positifs_matcher += 1
    for vrai in verite:
        if vrai not in predictions_matcher: faux_negatifs_matcher += 1
temps_matcher = time.perf_counter() - debut_matcher

# --- MÉTHODE 2 : POS TAGGING ---
debut_pos = time.perf_counter()
for texte, verite in jeu_de_test:
    doc = nlp_fr(texte) # Le pipeline complet s'exécute ici
    predictions_pos = [token.text for token in doc if token.pos_ == "PROPN"]

    # Calcul TP, FP, FN
    for pred in predictions_pos:
        if pred in verite: vrais_positifs_pos += 1
        else: faux_positifs_pos += 1
    for vrai in verite:
        if vrai not in predictions_pos: faux_negatifs_pos += 1
temps_pos = time.perf_counter() - debut_pos

# 4. FONCTION DE CALCUL DES MÉTRIQUES
def calculer_metriques(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    rappel = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * rappel) / (precision + rappel) if (precision + rappel) > 0 else 0
    return precision, rappel, f1

p_m, r_m, f1_m = calculer_metriques(vrais_positifs_matcher, faux_positifs_matcher, faux_negatifs_matcher)
p_p, r_p, f1_p = calculer_metriques(vrais_positifs_pos, faux_positifs_pos, faux_negatifs_pos)

# 5. AFFICHAGE
print(f"{'MÉTHODE':<17} | {'PRÉCISION':<10} | {'RAPPEL':<10} | {'F1-SCORE':<10} | {'TEMPS (s)'}")
print("-" * 68)
print(f"{'Matcher (Règles)':<17} | {p_m:.2f}       | {r_m:.2f}       | {f1_m:.2f}       | {temps_matcher:.5f}")
print(f"{'POS Tagging':<17} | {p_p:.2f}       | {r_p:.2f}       | {f1_p:.2f}       | {temps_pos:.5f}")

MÉTHODE           | PRÉCISION  | RAPPEL     | F1-SCORE   | TEMPS (s)
--------------------------------------------------------------------
Matcher (Règles)  | 0.70       | 0.97       | 0.82       | 0.00172
POS Tagging       | 0.93       | 0.78       | 0.85       | 0.11571


# 9. Question 9 — Analyse globale des corpus

Construisez les versions suivantes :

- C0 : texte original ;
- C1 : texte normalisé et lemmatisé ;
- C2 : texte normalisé, lemmatisé et débarrassé des mots vides.

Pour chacune, calculez nombre de tokens, nombre de formes distinctes, les trois mots les plus fréquents et les trois bigrammes les plus fréquents.

Comment le pré-traitement modifie-t-il la structure statistique des corpus ?

Comparez également le corpus français et le corpus italien.

Discutez les différences que vous observez et indiquez quelles différences peuvent être liées à la langue, au contenu des corpus ou au pré-traitement.



In [99]:
# ============================================================
# À COMPLÉTER — 9. Question 9 — Analyse globale des corpus
# ============================================================

import spacy
from collections import Counter

def analyser_structure_corpus(corpus_texts, nlp, nom_corpus):
    print(f"\n{'='*50}\nANALYSE DU CORPUS : {nom_corpus.upper()}\n{'='*50}")

    # Compteurs pour les statistiques globales
    c0_tokens, c1_tokens, c2_tokens = [], [], []
    c0_bigrams, c1_bigrams, c2_bigrams = [], [], []

    # Traitement optimisé par lots
    for doc in nlp.pipe(corpus_texts, disable=["ner"]): # Désactivation du NER pour accélérer
        doc_c0, doc_c1, doc_c2 = [], [], []

        for token in doc:
            # Filtrer les espaces vides qui faussent les statistiques
            if token.is_space:
                continue

            # C0 : Texte original
            doc_c0.append(token.text)

            # C1 : Normalisé (minuscules) et lemmatisé
            lemme_norm = token.lemma_.lower()
            doc_c1.append(lemme_norm)

            # C2 : Normalisé, lemmatisé, sans mots vides (ni ponctuation)
            if not token.is_stop and not token.is_punct:
                doc_c2.append(lemme_norm)

        # Ajout aux listes globales
        c0_tokens.extend(doc_c0)
        c1_tokens.extend(doc_c1)
        c2_tokens.extend(doc_c2)

        # Génération des bigrammes (intra-document pour éviter les liaisons artificielles)
        c0_bigrams.extend(zip(doc_c0[:-1], doc_c0[1:]))
        c1_bigrams.extend(zip(doc_c1[:-1], doc_c1[1:]))
        c2_bigrams.extend(zip(doc_c2[:-1], doc_c2[1:]))

    # Fonction utilitaire pour afficher les métriques d'une version
    def afficher_metriques(nom_version, tokens, bigrams):
        nb_tokens = len(tokens)
        nb_formes = len(set(tokens))
        top_mots = Counter(tokens).most_common(3)
        top_bigrammes = Counter(bigrams).most_common(3)

        print(f"\n--- {nom_version} ---")
        print(f"Nombre de tokens        : {nb_tokens}")
        print(f"Formes distinctes (vocab): {nb_formes}")
        print(f"3 Mots les plus fréquents: {top_mots}")
        print(f"3 Bigrammes fréquents    : {top_bigrammes}")

    afficher_metriques("C0 (Original)", c0_tokens, c0_bigrams)
    afficher_metriques("C1 (Normalisé + Lemmatisé)", c1_tokens, c1_bigrams)
    afficher_metriques("C2 (C1 sans mots vides/ponct)", c2_tokens, c2_bigrams)


analyser_structure_corpus(corpus_fr, nlp_fr, "Français")
analyser_structure_corpus(corpus_en, nlp_en, "Anglais")


ANALYSE DU CORPUS : FRANÇAIS

--- C0 (Original) ---
Nombre de tokens        : 5084033
Formes distinctes (vocab): 53193
3 Mots les plus fréquents: [(',', 242951), ('de', 234620), ('.', 161559)]
3 Bigrammes fréquents    : [(('de', 'la'), 42109), (('de', "l'"), 18933), (('à', 'la'), 15009)]

--- C1 (Normalisé + Lemmatisé) ---
Nombre de tokens        : 5084033
Formes distinctes (vocab): 30057
3 Mots les plus fréquents: [('le', 413408), ('de', 378095), (',', 242951)]
3 Bigrammes fréquents    : [(('de', 'le'), 62454), (('que', 'le'), 29436), ((',', 'le'), 26845)]

--- C2 (C1 sans mots vides/ponct) ---
Nombre de tokens        : 1998748
Formes distinctes (vocab): 29750
3 Mots les plus fréquents: [('européen', 23312), ('monsieur', 18959), ('commission', 18459)]
3 Bigrammes fréquents    : [(('monsieur', 'président'), 13323), (('union', 'européen'), 7884), (('état', 'membre'), 6310)]

ANALYSE DU CORPUS : ANGLAIS

--- C0 (Original) ---
Nombre de tokens        : 4718987
Formes distinctes (vocab): 

## Rappels utiles pour le notebook

Quelques attributs spaCy utiles : `token.text`, `token.lemma_`, `token.pos_`, `token.dep_`, `token.head`, `token.is_stop`, `token.is_punct`, `token.is_space`.


# Rendu

Votre archive `.zip` doit contenir :

1. le notebook Colab complété (ou son export `.ipynb`) ;
2. une synthèse d’une demi-page maximum de l’article.

Les réponses aux questions doivent se trouver directement dans le notebook.